# Person 2: Upload ZIP, preprocess, save to Drive

1. In Google Drive, download the existing **Clock_Drawing_Project** folder as a ZIP. It must include `raw_images/Data_DrawClocking` and `Dataset/Clock_Drawing_Master_Evaluation.xlsx`.
2. Run the upload cell below and select that ZIP. No Drive mount or shortcut is needed. Upload one ZIP containing all 400 images and the master workbook.
3. Run the remaining cells in order. They verify labels and images, generate the fixed 280/60/60 split, save 224 × 224 RGB PNGs, and create normalized float32 model arrays in Colab.
4. Run **Save results back to Drive**, sign in to the Google account that can edit the existing project, and wait for its verified completion message. It updates the existing `processed_images` and `splits` folders by their Drive IDs. If access is public, change General access to Restricted and rerun only that final cell.

Raw files are read only. The ZIP and local results stay in the temporary Colab runtime until saved back. The save cell does not upload raw images or alter sharing. Keep your original ZIP until the save finishes.

Split: SHA-256 ordering of `42:Clock_###`, first 280 train, next 60 validation, final 60 test. This is reproducible across Python versions and is not stratified. The four actual label columns, `Human_Total`, and `Notes` are preserved. The workbook guide uses different header names; labels are not renamed. Participant-level independence cannot be checked without participant IDs. Review any duplicate-image warnings before modeling.


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, json, warnings, importlib.metadata, io, zipfile, tempfile, shutil
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

def extract_project(zip_bytes, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
        members = archive.infolist()
        if sum(m.file_size for m in members) > 4 * 1024**3:
            raise ValueError('ZIP expands beyond 4 GB. Upload only the project dataset.')
        selected = []
        for member in members:
            relative = PurePosixPath(member.filename.replace(chr(92), '/'))
            if relative.is_absolute() or '..' in relative.parts or any(':' in p for p in relative.parts):
                raise ValueError('ZIP contains an unsafe path.')
            if '__MACOSX' in relative.parts or relative.name.startswith('._'):
                continue
            target = (destination / str(relative)).resolve()
            if not target.is_relative_to(destination):
                raise ValueError('ZIP contains a path outside the extraction folder.')
            selected.append((member, target))
        seen = set()
        for member, target in selected:
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if target in seen:
                raise ValueError(f'Duplicate ZIP member: {member.filename}')
            seen.add(target)
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open('wb') as output:
                shutil.copyfileobj(source, output)
    candidates = sorted({p.parent.parent for p in destination.rglob('Clock_Drawing_Master_Evaluation.xlsx')
                         if p.parent.name == 'Dataset'
                         and (p.parent.parent / 'raw_images' / 'Data_DrawClocking').is_dir()})
    if len(candidates) != 1:
        raise ValueError('Upload one complete project ZIP with Dataset/Clock_Drawing_Master_Evaluation.xlsx '
                         'and raw_images/Data_DrawClocking. Found ' + str(len(candidates)) + ' complete projects.')
    return candidates[0]

uploaded = files.upload()
if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith('.zip'):
    raise ValueError('Choose one ZIP of the complete Clock_Drawing_Project folder.')
WORKSPACE = Path(tempfile.mkdtemp(prefix='clock_preprocessing_', dir='/content'))
PROJECT = extract_project(next(iter(uploaded.values())), WORKSPACE)
del uploaded
RAW = PROJECT / 'raw_images' / 'Data_DrawClocking'
MASTER = PROJECT / 'Dataset' / 'Clock_Drawing_Master_Evaluation.xlsx'
PROCESSED = PROJECT / 'processed_images'
SPLIT_FILE = PROJECT / 'splits' / 'dataset_split.csv'
SEED = 42
LABEL_COLS = ['Closed_Circle', 'Number_correct_Position',
              'Includes_all_12_correct_Numbers', 'Places_Hands_in_correct_positions']
EXPECTED = {f'Clock_{i:03d}' for i in range(1, 401)}
print('Uploaded project:', PROJECT)
print({p: importlib.metadata.version(p) for p in ['pandas', 'numpy', 'Pillow', 'openpyxl']})


In [ ]:
# Locate the one worksheet with the actual labels, not the guide or summary.
worksheets = pd.read_excel(MASTER, sheet_name=None, engine='openpyxl')
required = ['Image_ID', *LABEL_COLS, 'Human_Total']
candidates = [(name, frame) for name, frame in worksheets.items()
              if set(required).issubset(frame.columns)]
assert len(candidates) == 1, f'Expected one labels sheet, found {[x[0] for x in candidates]}'
sheet_name, labels = candidates[0]
labels = labels.dropna(how='all').copy()
labels['Image_ID'] = labels['Image_ID'].astype(str).str.strip()
assert len(labels) == 400 and labels.Image_ID.is_unique, 'Missing or duplicate label rows.'
assert set(labels.Image_ID) == EXPECTED, 'Label IDs must be Clock_001 through Clock_400.'
assert labels[LABEL_COLS].isin([0, 1]).all().all(), 'Missing/non-binary human label.'
assert labels.Human_Total.notna().all(), 'Missing cached total: recalculate and save master in Excel.'
assert (labels[LABEL_COLS].sum(axis=1) == labels.Human_Total).all(), 'Human_Total mismatch.'
labels = labels.sort_values('Image_ID').reset_index(drop=True)
print(f'Validated {len(labels)} complete label rows in {sheet_name}.')


In [ ]:
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
image_paths = sorted(p for p in RAW.rglob('*') if p.is_file() and p.suffix.lower() in EXTENSIONS)
ignored = sorted(str(p.relative_to(RAW)) for p in RAW.rglob('*')
                 if p.is_file() and p.suffix.lower() not in EXTENSIONS)
by_id = {}
for path in image_paths:
    assert path.stem not in by_id, f'Duplicate image ID: {path.stem}'
    by_id[path.stem] = path
assert len(by_id) == 400 and set(by_id) == EXPECTED, (
    f'Missing images: {sorted(EXPECTED-set(by_id))}; extra IDs: {sorted(set(by_id)-EXPECTED)}')
assert set(by_id) == set(labels.Image_ID), 'Image/label mismatch.'
print('400 one-to-one image/label matches. Ignored non-images:', ignored)

def sha256_file(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def rgb_image(path):
    with Image.open(path) as im:
        assert getattr(im, 'n_frames', 1) == 1, f'Multiple frames: {path.name}'
        im.load()
        im = ImageOps.exif_transpose(im)
        # White background prevents transparency from turning into black pixels.
        rgba = im.convert('RGBA')
        background = Image.new('RGBA', rgba.size, (255, 255, 255, 255))
        return Image.alpha_composite(background, rgba).convert('RGB')

raw_hashes, pixel_groups, errors = {}, {}, []
for image_id, path in sorted(by_id.items()):
    try:
        with Image.open(path) as im:
            im.verify()
        rgb = rgb_image(path)
        assert min(rgb.size) > 0
        raw_hashes[image_id] = sha256_file(path)
        pixel_key = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
        pixel_groups.setdefault(pixel_key, []).append(image_id)
    except Exception as error:
        errors.append(f'{image_id}: {error}')
assert not errors, 'No outputs written. Image errors:\n' + '\n'.join(errors)
print('All 400 images opened and decoded successfully.')


In [ ]:
ordered = sorted(EXPECTED, key=lambda s: (hashlib.sha256(f'{SEED}:{s}'.encode()).hexdigest(), s))
assignment = {s: ('train' if i < 280 else 'validation' if i < 340 else 'test')
              for i, s in enumerate(ordered)}
expected_split = labels.copy()
expected_split['split'] = expected_split.Image_ID.map(assignment)
expected_split['raw_image_path'] = expected_split.Image_ID.map(lambda s: by_id[s].relative_to(PROJECT).as_posix())
expected_split['processed_image_path'] = expected_split.Image_ID.map(lambda s: f'processed_images/{s}.png')
expected_split['split_seed'] = SEED

if SPLIT_FILE.exists():
    split = pd.read_csv(SPLIT_FILE).sort_values('Image_ID').reset_index(drop=True)
    pd.testing.assert_frame_equal(split.fillna(''), expected_split.fillna(''), check_dtype=False)
    print('Existing CSV matches all master labels, paths, and deterministic assignments.')
else:
    split = expected_split
assert split['split'].value_counts().to_dict() == {'train': 280, 'validation': 60, 'test': 60}
assert split.Image_ID.is_unique
duplicate_groups = [ids for ids in pixel_groups.values() if len(ids) > 1]
cross_split_duplicates = [ids for ids in duplicate_groups if len({assignment[s] for s in ids}) > 1]
print('Exact duplicate pixel groups:', duplicate_groups)
if cross_split_duplicates:
    warnings.warn(f'Potential evaluation leakage: identical images span splits: {cross_split_duplicates}. '
                  'Review source/group information before training. Fixed assignments have not been changed.')
print(split['split'].value_counts())


## Process locally in Colab

Images are resized with Lanczos and centered on a white 224 × 224 canvas, preserving proportions. PNGs store RGB values 0–255. The following loader divides by 255 once to produce float32 model inputs in [0, 1]. Reruns reuse the verified split and replace only generated PNGs.


In [ ]:
PROCESSED.mkdir(parents=True, exist_ok=True)
SPLIT_FILE.parent.mkdir(parents=True, exist_ok=True)
for image_id in sorted(EXPECTED):
    rgb = rgb_image(by_id[image_id])
    processed = ImageOps.pad(rgb, (224, 224), method=Image.Resampling.LANCZOS,
                             color=(255, 255, 255), centering=(0.5, 0.5))
    target = PROCESSED / f'{image_id}.png'
    temporary = target.with_suffix('.tmp')
    processed.save(temporary, format='PNG')
    temporary.replace(target)
    with Image.open(target) as check:
        check.load()
        assert check.mode == 'RGB' and check.size == (224, 224), image_id

if not SPLIT_FILE.exists():
    temporary_csv = SPLIT_FILE.with_suffix('.tmp')
    split.to_csv(temporary_csv, index=False, lineterminator='\n')
    temporary_csv.replace(SPLIT_FILE)
assert all(sha256_file(by_id[s]) == raw_hashes[s] for s in EXPECTED), 'Raw source changed during run.'
saved = pd.read_csv(SPLIT_FILE)
pd.testing.assert_frame_equal(saved.fillna(''), split.fillna(''), check_dtype=False)
assert all((PROCESSED / f'{s}.png').is_file() for s in EXPECTED)
print('Verified 400 RGB 224x224 PNGs, saved split CSV, and unchanged raw image hashes.')


In [ ]:
# Model inputs: NHWC float32, range [0, 1]. No second normalization is needed.
# y contains the four human criteria; y_total contains the unchanged human total.
def load_partition(partition):
    assert partition in {'train', 'validation', 'test'}
    rows = split.loc[split['split'] == partition].sort_values('Image_ID')
    arrays = []
    for relative_path in rows.processed_image_path:
        with Image.open(PROJECT / relative_path) as im:
            assert im.mode == 'RGB' and im.size == (224, 224)
            arrays.append(np.asarray(im, dtype=np.float32) / np.float32(255.0))
    X = np.stack(arrays)
    y = rows[LABEL_COLS].to_numpy(dtype=np.float32)
    y_total = rows.Human_Total.to_numpy(dtype=np.float32)
    assert X.dtype == np.float32 and np.isfinite(X).all() and 0 <= X.min() <= X.max() <= 1
    return X, y, y_total, rows.Image_ID.to_numpy()

X_train, y_train, total_train, ids_train = load_partition('train')
X_validation, y_validation, total_validation, ids_validation = load_partition('validation')
X_test, y_test, total_test, ids_test = load_partition('test')
for name, X in [('train', X_train), ('validation', X_validation), ('test', X_test)]:
    print(name, X.shape, X.dtype, 'range:', float(X.min()), float(X.max()))
assert X_train.shape == (280, 224, 224, 3)
assert X_validation.shape == X_test.shape == (60, 224, 224, 3)
assert not (set(ids_train) & set(ids_validation) or set(ids_train) & set(ids_test)
            or set(ids_validation) & set(ids_test))
print('Preprocessing complete. Resolve any duplicate-image warnings before modeling.')


## Save results back to Drive

Run this after all processing cells succeed. Sign in with the account that can edit the existing project. This cell uses folder IDs, so mounted paths and shortcuts do not matter. It creates or updates each generated PNG and the split CSV without duplicating files. Existing split assignments and labels must agree with the current run. Public or domain-wide access blocks saving, but your processed results remain in Colab so you can fix sharing and rerun this cell.


In [ ]:
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
auth.authenticate_user()
api = build('drive', 'v3', cache_discovery=False)
PROJECT_ID = '1qLTle3GNsAPyKGXwOO-rhm2HAhfWoMNv'
DESTINATIONS = {'processed_images': '1j2F9IAD2jn4mhxrFMC5e2XWEsOXyFTo0',
                'splits': '1XTeENYHyBbGFM8X1Yz3xnzP5FTIwYGn7'}

def require_private(file_id, name):
    token = None
    permissions = []
    while True:
        response = api.permissions().list(fileId=file_id, supportsAllDrives=True,
            fields='nextPageToken,permissions(type,role)', pageToken=token).execute(num_retries=3)
        permissions.extend(response.get('permissions', []))
        token = response.get('nextPageToken')
        if not token: break
    if not permissions or any(p['type'] in {'anyone', 'domain'} for p in permissions):
        raise RuntimeError(f'{name} must be Restricted to the team before saving. '
                           'Change General access in Drive, then rerun this cell. Local results are retained.')

def list_children(folder_id):
    token, result = None, []
    while True:
        response = api.files().list(q=f"'{folder_id}' in parents and trashed = false",
            fields='nextPageToken,files(id,name,mimeType,md5Checksum)', pageSize=1000,
            supportsAllDrives=True, includeItemsFromAllDrives=True, pageToken=token).execute(num_retries=3)
        result.extend(response.get('files', []))
        token = response.get('nextPageToken')
        if not token: return result

require_private(PROJECT_ID, 'Clock_Drawing_Project')
indexes = {}
for name, fid in DESTINATIONS.items():
    metadata = api.files().get(fileId=fid, fields='id,name,parents,capabilities(canEdit)',
                               supportsAllDrives=True).execute(num_retries=3)
    assert PROJECT_ID in metadata.get('parents', []), f'{name} moved outside the project.'
    assert metadata.get('capabilities', {}).get('canEdit'), f'Your signed-in account cannot edit {name}.'
    require_private(fid, name)
    indexes[name] = list_children(fid)

targets = [('processed_images', PROCESSED / f'{s}.png', 'image/png') for s in sorted(EXPECTED)]
targets.append(('splits', SPLIT_FILE, 'text/csv'))
existing = {}
for folder, path, mime in targets:
    assert path.is_file(), f'Run processing first: missing {path.name}'
    matches = [f for f in indexes[folder] if f['name'] == path.name]
    assert len(matches) <= 1, f'Duplicate Drive filenames: {folder}/{path.name}. Resolve before saving.'
    if matches:
        item = matches[0]
        require_private(item['id'], folder + '/' + path.name)
        existing[(folder, path.name)] = item
        if path == SPLIT_FILE:
            data = api.files().get_media(fileId=item['id']).execute(num_retries=3)
            if isinstance(data, str): data = data.encode()
            cloud_split = pd.read_csv(io.BytesIO(data)).sort_values('Image_ID').reset_index(drop=True)
            local_split = pd.read_csv(path).sort_values('Image_ID').reset_index(drop=True)
            pd.testing.assert_frame_equal(cloud_split.fillna(''), local_split.fillna(''), check_dtype=False)

verified = 0
for folder, path, mime in targets:
    digest = hashlib.md5(path.read_bytes()).hexdigest()
    item = existing.get((folder, path.name))
    if item and item.get('md5Checksum') == digest:
        verified += 1
        continue
    media = MediaFileUpload(str(path), mimetype=mime, resumable=True)
    if item:
        request = api.files().update(fileId=item['id'], media_body=media,
            fields='id,name,md5Checksum', supportsAllDrives=True)
    else:
        request = api.files().create(body={'name': path.name, 'parents': [DESTINATIONS[folder]]},
            media_body=media, fields='id,name,md5Checksum', supportsAllDrives=True)
    response = None
    while response is None:
        _, response = request.next_chunk(num_retries=3)
    assert response.get('md5Checksum') == digest, f'Upload checksum mismatch: {path.name}'
    verified += 1
    if verified % 50 == 0: print(f'Uploaded/verified {verified}/401 files')

for name, fid in DESTINATIONS.items():
    saved = list_children(fid)
    for folder, path, _ in targets:
        if folder != name: continue
        matches = [f for f in saved if f['name'] == path.name]
        assert len(matches) == 1 and matches[0].get('md5Checksum') == hashlib.md5(path.read_bytes()).hexdigest()
print('SUCCESS: verified 400 processed PNGs and dataset_split.csv in the existing Drive project.')
print('Raw images and master workbook were not changed. Split: 280 train / 60 validation / 60 test.')


## Handoff

Person 3 can use `processed_images/Clock_###.png` and `splits/dataset_split.csv` in the original Drive project after the final success message. The model arrays are also available in this Colab session. Do not normalize them a second time. Reserve the test set for final evaluation and review duplicate-image warnings before training.

This notebook is delivered without executed outputs. Actual image validation, processing, and authenticated Drive saving happen when you run the cells. If the runtime resets before saving, upload your ZIP and rerun. No Drive mounting is required.
